# NLP Project - Insurance Reviews Analysis


In [1]:
# Imports
import os
import re
import pickle
from collections import Counter
import gdown

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

DATA_DIR = "data"
RAW_FILE = os.path.join(DATA_DIR, "insurance_reviews.zip")
CSV_FILE = os.path.join(DATA_DIR, "insurance_reviews.csv")
GDRIVE_URL = "https://drive.google.com/file/d/1_kg5JzAzntzLI6eGM3_vmUSoeWk7f8ip/view?usp=sharing"

gdown.download(url=GDRIVE_URL, output=RAW_FILE, quiet=False, fuzzy=True)

Downloading...
From: https://drive.google.com/uc?id=1_kg5JzAzntzLI6eGM3_vmUSoeWk7f8ip
To: c:\Users\sulta\Documents\NLP_project_no2\data\insurance_reviews.zip
100%|██████████| 12.0M/12.0M [00:01<00:00, 8.42MB/s]


'data\\insurance_reviews.zip'

In [2]:
import zipfile
import io

def load_reviews(path):
    dfs = []
    with zipfile.ZipFile(path, "r") as z:
        for name in z.namelist():
            with z.open(name) as f:
                dfs.append(pd.read_excel(io.BytesIO(f.read()), engine="openpyxl"))
    df = pd.concat(dfs, ignore_index=True)
    return df

df = load_reviews(RAW_FILE)

# Save to CSV
df.to_csv(CSV_FILE, index=False, encoding="utf-8")
print(f"DataFrame saved in {CSV_FILE}, ({len(df)} rows)")

df.head()

DataFrame saved in data\insurance_reviews.csv, (34435 rows)


,note,auteur,avis,assureur,produit,type,date_publication,date_exp,avis_en,avis_cor,avis_cor_en
0,1.0,maitre-en-colere-114722,Pour tous les maîtres des animaux\n L'assuranc...,Eca Assurances,animaux,train,25/05/2021,01/05/2021,For all masters of animals\n ECA insurance and...,NaN,NaN
1,5.0,fredo-102557,"25 ans chez AMV pour ma moto et mes voitures, ...",AMV,moto,train,13/01/2021,01/01/2021,Loading...,NaN,NaN
2,4.0,louisonne-f-130215,Je suis satisfait de votre prestation et servi...,L'olivier Assurance,auto,train,30/08/2021,01/08/2021,Loading...,NaN,NaN
3,3.0,stephane-p-128761,SATISFAIT ET FACILE DE SOUSCRIRE UN CONTRAT EN...,AMV,moto,train,20/08/2021,01/08/2021,Loading...,NaN,NaN
4,1.0,montad-137724,Je croyais qu'allianz était une bonne assuranc...,Allianz,auto,train,18/10/2021,01/10/2021,Loading...,NaN,NaN


In [3]:
missing_counts = df.isna().sum()
missing_counts = missing_counts[missing_counts > 0]
print("columns with missing values and number of missing rows:")
print(missing_counts)

columns with missing values and number of missing rows:
note           10331
auteur             1
avis_en            2
avis_cor       34000
avis_cor_en    34004
dtype: int64


## 1. Data Cleaning

Handle missing values, convert dates, and preprocess text (punctuation removal, stopwords, lemmatization).

In [ ]:
# Metrics before cleaning
n_before = len(df)
mean_note_before = df['note'].mean()
density_before = 1 - df.isna().sum().sum() / (n_before * len(df.columns))

# Remove rows with empty avis (French reviews)
df_clean = df.dropna(subset=['avis']).copy()
df_clean = df_clean[df_clean['avis'].astype(str).str.strip() != '']

# Remove rows with missing note (target for modeling)
df_clean = df_clean.dropna(subset=['note'])

# Fill auteur missing (1 row) with default
df_clean['auteur'] = df_clean['auteur'].fillna('unknown')

# Convert date_publication to datetime
df_clean['date_publication'] = pd.to_datetime(df_clean['date_publication'], format='%d/%m/%Y', errors='coerce')

# Drop rows with invalid dates if needed (optional)
df_clean = df_clean.dropna(subset=['date_publication'])

n_removed = n_before - len(df_clean)
print(f"Rows removed: {n_removed}")
print(f"Removal reasons: avis empty, note missing, invalid date")

In [ ]:
# Text preprocessing: use avis_cor if not empty, else avis
from string import punctuation
import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

FR_STOPWORDS = set(stopwords.words('french'))
DOMAIN_STOPWORDS = {'assurance', 'assureur', 'contrat', 'remboursement', 'client', 'service'}
STOPWORDS = FR_STOPWORDS | DOMAIN_STOPWORDS
stemmer = SnowballStemmer('french')

def preprocess_text(text):
    if pd.isna(text) or not str(text).strip():
        return ''
    text = str(text).lower()
    text = ''.join(c for c in text if c not in punctuation)
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return ' '.join(stemmer.stem(t) for t in tokens)

# Select source column: avis_cor if not empty, else avis
df_clean['avis_source'] = df_clean.apply(
    lambda r: r['avis_cor'] if pd.notna(r['avis_cor']) and str(r['avis_cor']).strip() else r['avis'],
    axis=1
)
df_clean['avis_traite'] = df_clean['avis_source'].apply(preprocess_text)
df_clean.head()

In [ ]:
# Metrics after cleaning
n_after = len(df_clean)
mean_note_after = df_clean['note'].mean()
density_after = 1 - df_clean.isna().sum().sum() / (n_after * len(df_clean.columns))

print("=== Cleaning metrics ===")
print(f"Before: n={n_before}, mean_note={mean_note_before:.3f}, density={density_before:.3f}")
print(f"After:  n={n_after}, mean_note={mean_note_after:.3f}, density={density_after:.3f}")
print(f"Rows removed: {n_removed} (motifs: avis empty, note missing, invalid date)")

df_cleaned = df_clean.copy()

## 2. NLP Tasks

### 2.1 Spell correction (on a fraction of the dataset)

In [ ]:
# Spell correction on a fraction of the dataset (scale parameter)
from spellchecker import SpellChecker

SPELL_SCALE = 0.01  # Process 1% of the dataset
spell = SpellChecker(language='fr')

sample_idx = df_cleaned.sample(frac=SPELL_SCALE, random_state=42).index
corrections_examples = []

for idx in sample_idx:
    text = df_cleaned.loc[idx, 'avis']
    if pd.isna(text) or not str(text).strip():
        continue
    words = str(text).split()
    corrected_words = []
    changed = False
    for word in words:
        clean_word = ''.join(c for c in word.lower() if c.isalpha())
        if clean_word and clean_word not in spell:
            candidates = spell.candidates(clean_word)
            if candidates:
                fix = spell.correction(clean_word)
                if fix and fix != clean_word:
                    corrected_words.append(fix)
                    changed = True
                    if len(corrections_examples) < 2:
                        corrections_examples.append((word, fix))
                else:
                    corrected_words.append(word)
            else:
                corrected_words.append(word)
        else:
            corrected_words.append(word)
    if changed:
        df_cleaned.loc[idx, 'avis_cor'] = ' '.join(corrected_words)

if corrections_examples:
    print("2 examples of spell corrections:")
    for orig, fixed in corrections_examples:
        print(f"  '{orig}' -> '{fixed}'")
else:
    print("No spell corrections found in sample.")

### 2.2 Translation FR -> EN (Hugging Face)

Translate only rows where avis_en == "Loading...". Use a small sample for demo.

In [ ]:
# Translation: only rows with avis_en == "Loading..."
from transformers import pipeline

TRANSLATE_SCALE = 0.005  # Small sample for demo (Helsinki model is fast)
translate_pipe = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en")

to_translate = df_cleaned[df_cleaned['avis_en'] == 'Loading...'].sample(
    frac=TRANSLATE_SCALE, random_state=42
)
for idx in to_translate.index:
    text = df_cleaned.loc[idx, 'avis']
    if pd.isna(text) or not str(text).strip() or len(str(text)) > 512:
        continue
    result = translate_pipe(str(text)[:512])[0]
    df_cleaned.loc[idx, 'avis_en'] = result['translation_text']

print(f"Translated {len(to_translate)} reviews.")

In [ ]:
# Refresh avis_source and avis_traite after spell correction (use avis_cor when available)
df_cleaned['avis_source'] = df_cleaned.apply(
    lambda r: r['avis_cor'] if pd.notna(r['avis_cor']) and str(r['avis_cor']).strip() else r['avis'],
    axis=1
)
df_cleaned['avis_traite'] = df_cleaned['avis_source'].apply(preprocess_text)

## 3. Exploration

N-gram analysis before/after cleaning, rating distribution, temporal evolution, word count by rating.

In [ ]:
# N-grams before and after cleaning
from sklearn.feature_extraction.text import CountVectorizer

# Before: raw avis (sample for speed)
texts_before = df_cleaned['avis'].dropna().astype(str).sample(5000, random_state=42)
vec_before = CountVectorizer(ngram_range=(2, 2), max_features=20)
bigrams_before = vec_before.fit_transform(texts_before)
print("Top 10 bigrams BEFORE cleaning:")
for w, c in sorted(zip(vec_before.get_feature_names_out(), bigrams_before.sum(axis=0).A1), key=lambda x: -x[1])[:10]:
    print(f"  {w}: {int(c)}")

# After: avis_traite
texts_after = df_cleaned['avis_traite'].dropna()
texts_after = texts_after[texts_after.str.len() > 0].sample(5000, random_state=42)
vec_after = CountVectorizer(ngram_range=(2, 2), max_features=20)
bigrams_after = vec_after.fit_transform(texts_after)
print("\nTop 10 bigrams AFTER cleaning:")
for w, c in sorted(zip(vec_after.get_feature_names_out(), bigrams_after.sum(axis=0).A1), key=lambda x: -x[1])[:10]:
    print(f"  {w}: {int(c)}")

In [ ]:
# Distribution of ratings
plt.figure(figsize=(8, 4))
sns.countplot(data=df_cleaned, x='note', order=sorted(df_cleaned['note'].dropna().unique()))
plt.title("Distribution of ratings")
plt.xlabel("Note")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Temporal evolution of ratings by insurer (top insurers by volume)
top_assureurs = df_cleaned['assureur'].value_counts().head(6).index
df_top = df_cleaned[df_cleaned['assureur'].isin(top_assureurs)]
monthly = df_top.groupby([pd.Grouper(key='date_publication', freq='M'), 'assureur'])['note'].mean().unstack()
monthly.plot(figsize=(12, 5), title="Monthly average rating by insurer")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
# Word count per review by rating
df_cleaned['word_count'] = df_cleaned['avis_traite'].str.split().str.len().fillna(0)
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_cleaned, x='note', y='word_count', order=sorted(df_cleaned['note'].dropna().unique()))
plt.title("Word count per review by rating")
plt.xlabel("Note")
plt.ylabel("Word count")
plt.tight_layout()
plt.show()

## 4. Embeddings (Word2Vec)

Train Word2Vec, visualize with PCA/t-SNE and TensorBoard, compute cosine similarity.

In [ ]:
# Word2Vec on tokenized reviews
from gensim.models import Word2Vec

sentences = [str(t).split() for t in df_cleaned['avis_traite'] if pd.notna(t) and len(str(t).strip()) > 0]
sentences = [s for s in sentences if len(s) >= 2]

w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4, seed=42)
print(f"Vocabulary size: {len(w2v_model.wv)}")

In [ ]:
# Cosine similarity: most similar words to reference terms
ref_words = ['remboursement', 'rembours', 'client', 'servic', 'contrat', 'souscr']
for word in ref_words:
    if word in w2v_model.wv:
        sims = w2v_model.wv.most_similar(word, topn=5)
        print(f"{word}: {[w for w, _ in sims]}")

In [ ]:
# Visualization with Matplotlib (PCA)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# Get vectors for a sample of words
words = list(w2v_model.wv.index_to_key)[:200]
vectors = np.array([w2v_model.wv[w] for w in words])

# Reduce to 2D with PCA (faster than t-SNE for demo)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(vectors)

plt.figure(figsize=(12, 8))
plt.scatter(coords[:, 0], coords[:, 1], alpha=0.6)
for i, w in enumerate(words):
    plt.annotate(w, (coords[i, 0], coords[i, 1]), fontsize=8, alpha=0.8)
plt.title("Word2Vec - PCA projection")
plt.tight_layout()
plt.show()

In [ ]:
# Save for TensorBoard Projector (vectors + metadata)
vec_path = os.path.join(DATA_DIR, "w2v_vectors.tsv")
meta_path = os.path.join(DATA_DIR, "w2v_metadata.tsv")
with open(vec_path, "w", encoding="utf-8") as vf, open(meta_path, "w", encoding="utf-8") as mf:
    mf.write("word\n")
    for w in w2v_model.wv.index_to_key:
        vec = w2v_model.wv[w]
        vf.write("\t".join(map(str, vec)) + "\n")
        mf.write(f"{w}\n")
print("Saved w2v_vectors.tsv and w2v_metadata.tsv for TensorBoard Projector.")

## 5. Topic Modelling and Thematics

LDA for topic discovery, deterministic thematic detection with keywords + Word2Vec, supervised topic prediction.

In [ ]:
# LDA for topic exploration
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore

documents = [str(t).split() for t in df_cleaned['avis_traite'] if pd.notna(t) and len(str(t).strip()) > 0]
dictionary = Dictionary(documents)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(doc) for doc in documents]

lda = LdaMulticore(corpus, id2word=dictionary, num_topics=6, workers=4, random_state=42)
print("LDA topics (top 5 words per topic):")
for tid in range(6):
    print(f"  Topic {tid}: {[w for w, _ in lda.show_topic(tid, 5)]}")

In [ ]:
# Deterministic thematic detection: keywords + Word2Vec variants
THEMATIQUES = {
    'remboursements': ['rembours', 'prestation', 'paiement', 'indemn', 'franchis'],
    'service_client': ['mail', 'telephon', 'client', 'contact', 'accueil', 'reactiv'],
    'gestion_contrat': ['souscr', 'contrat', 'resili', 'prélev', 'changement'],
}

# Enrich with Word2Vec similar words
for theme, keywords in list(THEMATIQUES.items()):
    for kw in keywords[:2]:  # Limit to avoid too many
        if kw in w2v_model.wv:
            sims = w2v_model.wv.most_similar(kw, topn=3)
            for w, _ in sims:
                if w not in keywords:
                    THEMATIQUES[theme].append(w)

def detect_thematique(tokens):
    themes = []
    for theme, kws in THEMATIQUES.items():
        if any(kw in tokens for kw in kws):
            themes.append(theme)
    return themes[0] if themes else 'autre'

df_cleaned['tokens'] = df_cleaned['avis_traite'].apply(lambda x: str(x).split() if pd.notna(x) else [])
df_cleaned['thematique'] = df_cleaned['tokens'].apply(detect_thematique)

In [ ]:
# Average rating per thematic
print("Average rating by thematic:")
print(df_cleaned.groupby('thematique')['note'].agg(['mean', 'count']))

In [ ]:
# Supervised topic prediction: train classifier on TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X = df_cleaned['avis_traite'].fillna('')
y = df_cleaned['thematique']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf = tfidf.transform(X_test)

clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(X_train_tf, y_train)
y_pred = clf.predict(X_test_tf)

print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=clf.classes_))

## 6. Sentiment Analysis

Hugging Face pipeline on English translations (or French if multilingual model).

In [ ]:
# Sentiment analysis: use avis_en when available (not Loading...), else sample of avis for multilingual
sentiment_pipe = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

def get_text_for_sentiment(row):
    if pd.notna(row['avis_en']) and str(row['avis_en']).strip() != '' and 'Loading' not in str(row['avis_en']):
        return str(row['avis_en'])[:512]
    return None

df_sent = df_cleaned.copy()
df_sent['text_sent'] = df_sent.apply(get_text_for_sentiment, axis=1)
df_sent = df_sent[df_sent['text_sent'].notna()]
df_sent = df_sent.sample(min(500, len(df_sent)), random_state=42)  # Sample for speed

if len(df_sent) > 0:
    results = sentiment_pipe(df_sent['text_sent'].tolist())
    df_sent['sentiment_label'] = [r['label'] for r in results]
    df_sent['sentiment_score'] = [r['score'] for r in results]
else:
    df_sent['sentiment_label'] = []
    df_sent['sentiment_score'] = []

In [ ]:
# Compare sentiment with notes
if len(df_sent) > 0 and 'sentiment_label' in df_sent.columns:
    print("Sentiment vs Note (sample):")
    print(df_sent.groupby(['note', 'sentiment_label']).size().unstack(fill_value=0))
else:
    print("No translated reviews for sentiment analysis.")

## 7. Interpretation

Error analysis and SHAP for explaining predictions.

In [ ]:
# Error analysis: misclassified examples (thematic classifier)
wrong_mask = y_test != y_pred
wrong_df = pd.DataFrame({
    'text': X_test[wrong_mask],
    'true': y_test[wrong_mask],
    'pred': y_pred[wrong_mask]
})
print("Sample of misclassified reviews (true vs pred):")
for i, row in wrong_df.head(5).iterrows():
    print(f"  True: {row['true']} | Pred: {row['pred']}")
    print(f"  Text: {row['text'][:150]}...")

In [ ]:
# SHAP for explaining thematic predictions
import shap

explainer = shap.LinearExplainer(clf, X_train_tf)
shap_values = explainer.shap_values(X_test_tf[:100])
# For multiclass, shap_values is a list; take first class for summary
sv = shap_values[0] if isinstance(shap_values, list) else shap_values
shap.summary_plot(sv, X_test_tf[:100], feature_names=tfidf.get_feature_names_out(), show=False)
plt.title("SHAP - Top features for thematic prediction")
plt.tight_layout()
plt.show()

## 8. Application - Save artifacts for Streamlit

Save models and vectorizers for the Streamlit app.

In [ ]:
# Save artifacts for Streamlit app
ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

with open(os.path.join(ARTIFACTS_DIR, "preprocess.pkl"), "wb") as f:
    pickle.dump({"stopwords": STOPWORDS, "stemmer": stemmer}, f)
with open(os.path.join(ARTIFACTS_DIR, "tfidf.pkl"), "wb") as f:
    pickle.dump(tfidf, f)
with open(os.path.join(ARTIFACTS_DIR, "clf_thematic.pkl"), "wb") as f:
    pickle.dump(clf, f)
w2v_model.save(os.path.join(ARTIFACTS_DIR, "w2v.model"))
df_cleaned.to_csv(os.path.join(DATA_DIR, "insurance_reviews_cleaned.csv"), index=False, encoding="utf-8")

print("Artifacts saved.")

## Run Streamlit app

Execute in terminal: `streamlit run app_streamlit.py`